# Anomaly detection for payment fraud

Compares a supervised Random Forest (with SMOTE) against three unsupervised detectors: Isolation Forest, One-Class SVM and Local Outlier Factor, on the Kaggle *Credit Card Fraud Detection* dataset.

Download `creditcard.csv` from [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) and put it next to this notebook. Results and discussion are in the README.

In [ ]:
!pip install imbalanced-learn -q

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                              roc_auc_score, average_precision_score)
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('creditcard.csv')
print(df.shape)
print(df['Class'].value_counts())

In [ ]:
feature_cols = [c for c in df.columns if c != 'Class']
X = df[feature_cols]
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, fraud={y_train.sum()}")
print(f"Test:  {X_test.shape}, fraud={y_test.sum()}")

In [ ]:
X_train_tune, X_val, y_train_tune, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)
print(f"For tuning: {X_train_tune.shape}, fraud={y_train_tune.sum()}")
print(f"Validation: {X_val.shape}, fraud={y_val.sum()}")

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print(f"After SMOTE: {X_train_smote.shape}, fraud={y_train_smote.sum()}")

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_smote, y_train_smote)

rf_scores = rf.predict_proba(X_test)[:, 1]   # fraud probability, already oriented correctly
rf_preds  = rf.predict(X_test)

In [ ]:
from sklearn.metrics import f1_score

best_contamination, best_f1_if = None, -1
for c in [0.001, 0.005, 0.01, 0.02]:
    iso_tmp = IsolationForest(n_estimators=100, contamination=c,
                               random_state=RANDOM_STATE, n_jobs=-1)
    iso_tmp.fit(X_train_tune)
    preds = np.where(iso_tmp.predict(X_val) == -1, 1, 0)
    f1 = f1_score(y_val, preds, zero_division=0)
    print(f"contamination={c}: F1={f1:.4f}")
    if f1 > best_f1_if:
        best_contamination, best_f1_if = c, f1

print(f"\nBest contamination: {best_contamination} (F1={best_f1_if:.4f})")

In [ ]:
iso = IsolationForest(n_estimators=100, contamination=best_contamination,
                       random_state=RANDOM_STATE, n_jobs=-1)
iso.fit(X_train)
iso_scores = -iso.decision_function(X_test)
iso_preds  = np.where(iso.predict(X_test) == -1, 1, 0)

In [ ]:
SUBSAMPLE_SIZE = 25000

X_train_legal = X_train[y_train == 0]
X_sub = X_train_legal.sample(n=SUBSAMPLE_SIZE, random_state=RANDOM_STATE)

scaler = StandardScaler()
X_sub_scaled  = scaler.fit_transform(X_sub)
X_test_scaled = scaler.transform(X_test)   # same scaler, fitted on training data only

In [ ]:
X_train_tune_legal = X_train_tune[y_train_tune == 0]
X_sub_tune = X_train_tune_legal.sample(n=min(25000, len(X_train_tune_legal)), random_state=RANDOM_STATE)

scaler_tune = StandardScaler()
X_sub_tune_scaled = scaler_tune.fit_transform(X_sub_tune)
X_val_scaled = scaler_tune.transform(X_val)

best_nu, best_f1_ocsvm = None, -1
for nu_val in [0.005, 0.01, 0.02, 0.05]:
    ocsvm_tmp = OneClassSVM(kernel='rbf', nu=nu_val, gamma='scale')
    ocsvm_tmp.fit(X_sub_tune_scaled)
    preds = np.where(ocsvm_tmp.predict(X_val_scaled) == -1, 1, 0)
    f1 = f1_score(y_val, preds, zero_division=0)
    print(f"nu={nu_val}: F1={f1:.4f}")
    if f1 > best_f1_ocsvm:
        best_nu, best_f1_ocsvm = nu_val, f1

print(f"\nBest nu: {best_nu} (F1={best_f1_ocsvm:.4f})")

In [ ]:
ocsvm = OneClassSVM(kernel='rbf', nu=best_nu, gamma='scale')
ocsvm.fit(X_sub_scaled)   # same X_sub as above
ocsvm_scores = -ocsvm.decision_function(X_test_scaled)
ocsvm_preds  = np.where(ocsvm.predict(X_test_scaled) == -1, 1, 0)

In [ ]:
best_k, best_f1_lof = None, -1
for k in [10, 20, 35, 50]:
    lof_tmp = LocalOutlierFactor(n_neighbors=k, novelty=True, contamination='auto')
    lof_tmp.fit(X_sub_tune_scaled)   # same subsample as for OCSVM tuning
    preds = np.where(lof_tmp.predict(X_val_scaled) == -1, 1, 0)
    f1 = f1_score(y_val, preds, zero_division=0)
    print(f"n_neighbors={k}: F1={f1:.4f}")
    if f1 > best_f1_lof:
        best_k, best_f1_lof = k, f1

print(f"\nBest n_neighbors: {best_k} (F1={best_f1_lof:.4f})")

In [ ]:
lof = LocalOutlierFactor(n_neighbors=best_k, novelty=True, contamination='auto')
lof.fit(X_sub_scaled)
lof_scores = -lof.decision_function(X_test_scaled)
lof_preds  = np.where(lof.predict(X_test_scaled) == -1, 1, 0)

In [ ]:
def evaluate(name, y_true, y_pred, y_score):
    return {
        'Model': name,
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_score),
        'PR-AUC': average_precision_score(y_true, y_score),
    }

results = pd.DataFrame([
    evaluate('Random Forest',    y_test, rf_preds,    rf_scores),
    evaluate('Isolation Forest', y_test, iso_preds,   iso_scores),
    evaluate('One-Class SVM',    y_test, ocsvm_preds, ocsvm_scores),
    evaluate('LOF',              y_test, lof_preds,   lof_scores),
])
results = results.round(4)
print(results.to_string(index=False))

results.to_csv('comparison_results.csv', index=False)

try:  # download the file when running in Google Colab
    from google.colab import files
    files.download('comparison_results.csv')
except ImportError:
    pass

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

# --- colours used throughout the thesis ---
navy = '#1B3A5C'
cmap_custom = plt.cm.Blues

models_data = [
    ('Random Forest',    rf_preds),
    ('Isolation Forest', iso_preds),
    ('One-Class SVM',    ocsvm_preds),
    ('LOF',              lof_preds),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 10), dpi=150)
axes = axes.flatten()

labels = ['Legitimate', 'Fraud']

for ax, (name, preds) in zip(axes, models_data):
    cm = confusion_matrix(y_test, preds)                    # raw counts
    cm_norm = confusion_matrix(y_test, preds, normalize='true')  # for colour (share of each row)

    im = ax.imshow(cm_norm, cmap=cmap_custom, vmin=0, vmax=1)

    # axis labels
    ax.set_xticks([0, 1]); ax.set_xticklabels(labels, fontsize=10)
    ax.set_yticks([0, 1]); ax.set_yticklabels(labels, fontsize=10)
    ax.set_xlabel('Predicted class', fontsize=9.5)
    ax.set_ylabel('Actual class', fontsize=9.5)
    ax.set_title(name, fontsize=13, fontweight='bold', color=navy, pad=10)

    # annotations: count plus percentage in brackets
    for i in range(2):
        for j in range(2):
            count = cm[i, j]
            pct = cm_norm[i, j] * 100
            text_color = 'white' if cm_norm[i, j] > 0.5 else '#1E2D28'
            ax.text(j, i, f'{count:,}\n({pct:.1f}%)'.replace(',', ' '),
                     ha='center', va='center', fontsize=10.5,
                     color=text_color, fontweight='bold')

    for spine in ax.spines.values():
        spine.set_visible(False)

fig.suptitle('Confusion matrices of the compared algorithms (test set)',
              fontsize=15.5, fontweight='bold', y=1.0)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

try:  # download the file when running in Google Colab
    from google.colab import files
    files.download('confusion_matrices.png')
except ImportError:
    pass

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score

navy_dark = '#1B3A5C'
navy_mid  = '#3F78AB'
amber     = '#C98A2C'
red       = '#B4423C'

# Anomaly scores of each model, with its colour on the chart
models_scores = [
    ('Random Forest',    rf_scores,    navy_dark),
    ('Isolation Forest', iso_scores,   navy_mid),
    ('One-Class SVM',    ocsvm_scores, amber),
    ('LOF',              lof_scores,   red),
]

fig, ax = plt.subplots(figsize=(8, 6), dpi=150)

# PR curves
for name, scores, color in models_scores:
    precision, recall, _ = precision_recall_curve(y_test, scores)
    pr_auc = average_precision_score(y_test, scores)

    ax.plot(
        recall,
        precision,
        label=f'{name} (PR-AUC = {pr_auc:.3f})',
        color=color,
        linewidth=2.2
    )

# Baseline: a random classifier
baseline = y_test.sum() / len(y_test)

ax.axhline(
    baseline,
    color='#8A94A0',
    linestyle=(0, (5, 3)),
    linewidth=1.3,
    label=f'Random classifier ({baseline:.4f})'
)

# Axis labels
ax.set_xlabel('Recall', fontsize=11.5)
ax.set_ylabel('Precision', fontsize=11.5)

# Title
ax.set_title(
    'Precision-recall curves of the compared algorithms',
    fontsize=13.5,
    fontweight='bold',
    color='#22303C',
    pad=80  # room for the three-row legend above the plot
)

# Axis range
ax.set_xlim(0, 1.02)
ax.set_ylim(0, 1.02)

# Legend above the plot
ax.legend(
    loc='lower center',
    bbox_to_anchor=(0.5, 1.03),
    ncol=2,
    fontsize=9.5,
    frameon=False,
    columnspacing=1.5,
    handlelength=2.5
)

# Remove the top and right frame
ax.spines[['top', 'right']].set_visible(False)

# Grid
ax.grid(
    True,
    color='#E3E8EC',
    linewidth=0.7
)

# keep the layout from squeezing the top
plt.tight_layout()

# Save
plt.savefig(
    'pr_curves.png',
    dpi=150,
    bbox_inches='tight'
)

plt.show()

try:  # download the file when running in Google Colab
    from google.colab import files
    files.download('pr_curves.png')
except ImportError:
    pass